## ESI

* https://appliedsciences.nasa.gov/sites/default/files/2019-09/bluHain%20-%20Development%20of%20a%20Global%20Evaporative%20Stress%20Index%20Based%20on%20Thermal%20and%20Microwave%20LST%20Towards%20Improved%20Monitoring%20of%20Agricultural%20Drought.pdf
* https://www.drought.gov/data-maps-tools/evaporative-stress-index-esi 
* https://lpdaac.usgs.gov/documents/340/ECO4ESIALEXIU_ATBD_V1.pdf 

The Evaporative Stress Index (ESI) is available in two primary time scales: the 4-week and 12-week versions. Each serves different purposes in monitoring evapotranspiration (ET) anomalies and assessing agricultural drought conditions.

Differences Between ESI's 4-Week and 12-Week Time Scales
4-Week ESI (ESI_4wk)
* Time Frame: The 4-week ESI is calculated using a rolling window of daily ET data over the past four weeks. This allows for a more immediate response to changes in environmental conditions.
* Sensitivity: It is particularly sensitive to short-term fluctuations in moisture availability and can quickly capture rapid changes in crop stress, making it useful for identifying flash droughts.
* Application: Ideal for short-term agricultural decision-making, such as irrigation adjustments or immediate crop management strategies. It provides timely insights into current stress levels affecting crops.

12-Week ESI (ESI_12wk)
* Time Frame: The 12-week ESI aggregates data over a longer period, providing a broader view of ET anomalies over the past three months.
* Sensitivity: This version smooths out short-term variability, focusing instead on medium-term trends in moisture conditions. It is less reactive to immediate changes but offers a more stable assessment of overall water stress.
* Application: Useful for long-term planning and analysis, such as evaluating seasonal trends in crop health and making decisions related to crop rotation or long-term water resource management.


In summary, the choice between the 4-week and 12-week ESI depends on the specific needs of agricultural stakeholders—whether they require rapid insights into current conditions or a broader understanding of moisture trends over time.


## Soil Moisture 
* https://developers.google.com/earth-engine/datasets/catalog/NASA_GLDAS_V022_CLSM_G025_DA1D#bands

In [1]:
import os
import glob
import json
import sys
import time

import pandas as pd
import numpy as np
import rasterio
import rasterstats
import rioxarray as rxr
import geopandas as gpd
import geemap
from osgeo import gdal

dir = 'G:/My Drive/Work/ADB/SAR/'
sys.path.append('G:/My Drive/Work/ADB/SAR/src/gee_s1_ard/python-api')
import border_noise_correction as bnc
import speckle_filter as sf
import terrain_flattening as trf
from src.dhelper import get_month_range, get_last_day_of_month_from_Ym, get_prev_ndays

import ee
# Trigger the authentication flow.
try:
    ee.Initialize(project='ee-sonle96')
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project='ee-sonle96')

In [2]:
crs = 'EPSG:4326'
scale=100

START_YM = '2017-01'
END_YM = '2022-12'
months = get_month_range(START_YM, END_YM, input_format='%Y-%m', output_format='%Y-%m')

with open("D:\Work\ADB\SAR\shapefiles\JAXA_subtiles_full.json", 'r') as json_file:
    jaxa_subtiles = json.load(json_file)

# chosen_tiles = ['N10E105_0_1', 'N10E105_1_1', 'N10E106_0_1', 'N10E105_1_0','N09E105_1_0', 'N10E104_1_1', 'N09E105_0_0', 'N09E105_1_0', 'N09E105_0_1', 'N09E105_1_1', 'N10E105_0_0', 'N10E106_0_0']
chosen_tiles = ['N09E106_0_0', 'N09E105_0_0', 'N09E105_0_0', 'N10E105_0_1', 'N10E105_0_0', 'N10E105_1_0',
                'N10E106_0_0', 'N10E106_1_0', 'N10E106_1_1']   


wheat = ee.Image('projects/ee-sonle96/assets/worldcereal_wintercereals_vn')
wheat_mask = wheat.select('b1').eq(100)

maize = ee.Image('projects/ee-sonle96/assets/worldcereal_maize_vn')
maize_mask = maize.select('b1').eq(100)

crop_type = 'Maize'

if crop_type == 'Wheat':
    def mask_crop_map(image):
        return image.updateMask(wheat_mask)

if crop_type == 'Maize':
    def mask_crop_map(image):
        return image.updateMask(maize_mask)


<>:8: SyntaxWarning: invalid escape sequence '\W'
<>:8: SyntaxWarning: invalid escape sequence '\W'


C:\Users\sonle\AppData\Local\Temp\ipykernel_39728\388157374.py:8: SyntaxWarning: invalid escape sequence '\W'
  with open("D:\Work\ADB\SAR\shapefiles\JAXA_subtiles_full.json", 'r') as json_file:


In [17]:
print(f"CRS: {crs}")
print(f"scale: {scale}")
print(f"Start: {START_YM}. End: {END_YM}")

for tile in chosen_tiles[:1]:
    print(f"  {tile}")
    bbox = jaxa_subtiles[tile]
    ROI = ee.Geometry.BBox(bbox[0],
                            bbox[1],
                            bbox[2],
                            bbox[3])

    for month in months[5:6]:
        print(f"    {month}")

        START_DATE = month + '-01'
        STOP_DATE = get_last_day_of_month_from_Ym(month,
                                                input_format='%Y-%m',
                                                output_format='%Y-%m-%d')
        
        esi_4wk_ic = ee.ImageCollection('projects/climate-engine/esi/4wk')\
                       .filterDate(START_DATE, STOP_DATE)\
                    #    .map(mask_crop_map)
        
        esi_4wk_mean = esi_4wk_ic.select('ESI')\
                                 .mean()\
                                 .rename('ESI_4wk_mean')\
                                 .reproject(crs=crs, scale=scale)\
                                 .clip(ROI)\
                                 .updateMask(wheat_mask)
        
        esi_4wk_min = esi_4wk_ic.select('ESI')\
                                 .min()\
                                 .rename('ESI_4wk_min')\
                                 .reproject(crs=crs, scale=scale)\
                                 .clip(ROI)

        esi_4wk_max = esi_4wk_ic.select('ESI')\
                                 .max()\
                                 .rename('ESI_4wk_max')\
                                 .reproject(crs=crs, scale=scale)\
                                 .clip(ROI)
        
        
        modis_et_d_ic = ee.ImageCollection('projects/earthengine-legacy/assets/projects/usgs-ssebop/modis_et_v5_dekadal')\
                          .filterDate(START_DATE, STOP_DATE)\
                          .map(mask_crop_map)
        
        et_mean = modis_et_d_ic.select('et')\
                               .mean()\
                               .rename('ET_mean')\
                               .reproject(crs=crs, scale=scale)\
                               .clip(ROI)
        
        et_min = modis_et_d_ic.select('et')\
                               .min()\
                               .rename('ET_min')\
                               .reproject(crs=crs, scale=scale)\
                               .clip(ROI)
        
        et_max = modis_et_d_ic.select('et')\
                               .max()\
                               .rename('ET_max')\
                               .reproject(crs=crs, scale=scale)\
                               .clip(ROI)
        
        smap_l4_ic = ee.ImageCollection('NASA/SMAP/SPL4SMGP/007')\
                       .filterDate(START_DATE, STOP_DATE)\
                       .map(mask_crop_map)
        
        sm_root_mean = smap_l4_ic.select('sm_rootzone')\
                                 .mean()\
                                 .rename('SM_ROOT_mean')\
                                 .reproject(crs=crs, scale=scale)\
                                 .clip(ROI)
        
        sm_root_min = smap_l4_ic.select('sm_rootzone')\
                                 .min()\
                                 .rename('SM_ROOT_min')\
                                 .reproject(crs=crs, scale=scale)\
                                 .clip(ROI)
        
        sm_root_max = smap_l4_ic.select('sm_rootzone')\
                                 .max()\
                                 .rename('SM_ROOT_max')\
                                 .reproject(crs=crs, scale=scale)\
                                 .clip(ROI)
        
        
        MCD15A3H = ee.ImageCollection("MODIS/061/MCD15A3H")\
                     .filterDate(START_DATE, STOP_DATE)\
                     .map(mask_crop_map)

        def extract_bits(image, bit_start, bit_end):
            num_bits = bit_end - bit_start + 1
            mask = (1 << num_bits) - 1
            return image.rightShift(bit_start).bitwiseAnd(mask)

        def apply_bitmask_terra(image):

            qc_band = image.select('FparLai_QC')

            # Extract values for the specific bit ranges
            # modland_qc = extract_bits(qc_band, 0, 0).eq(0)  # Bit 0 should be 0 (Good quality - main algorithm with or without saturation)
            # detector = extract_bits(qc_band, 2, 2).eq(0)    # Bit 2 should be 0
            # cloud_state = extract_bits(qc_band, 3, 4).eq(0) # Bits 3-4 should be 0 (clear)
            scf_qc = extract_bits(qc_band, 5, 7).neq(4)     # Bits 5-7 should not be 4 (Pixel not produced at all, value couldn''t be retrieved)

            # combined_mask = modland_qc.And(detector).And(cloud_state).And(scf_qc)

            # Sensor
            # terra_mask = combined_mask.And(extract_bits(qc_band, 1, 1).eq(0)) 
            terra_mask = scf_qc.And(extract_bits(qc_band, 1, 1).eq(0)) 

            return image.updateMask(terra_mask)

        def apply_bitmask_aqua(image):

            qc_band = image.select('FparLai_QC')

            # Extract values for the specific bit ranges
            # modland_qc = extract_bits(qc_band, 0, 0).eq(0)  # Bit 0 should be 0 (Good quality - main algorithm with or without saturation)
            # detector = extract_bits(qc_band, 2, 2).eq(0)    # Bit 2 should be 0
            # cloud_state = extract_bits(qc_band, 3, 4).eq(0) # Bits 3-4 should be 0 (clear)
            scf_qc = extract_bits(qc_band, 5, 7).neq(4)     # Bits 5-7 should not be 4 (Pixel not produced at all, value couldn''t be retrieved)

            # combined_mask = modland_qc.And(detector).And(cloud_state).And(scf_qc)
            # aqua_mask =  combined_mask.And(extract_bits(qc_band, 1, 1).eq(1)) 
            aqua_mask =  scf_qc.And(extract_bits(qc_band, 1, 1).eq(1)) 

            return image.updateMask(aqua_mask)

        dataset_terra = MCD15A3H.map(apply_bitmask_terra)

        fpar_terra_mean = dataset_terra.select('Fpar')\
                                       .mean()\
                                       .multiply(0.01)\
                                       .rename('FPAR_Terra_mean')\
                                       .reproject(crs=crs, scale=scale)\
                                       .clip(ROI)
        
        fpar_terra_min = dataset_terra.select('Fpar')\
                                       .min()\
                                       .multiply(0.01)\
                                       .rename('FPAR_Terra_min')\
                                       .reproject(crs=crs, scale=scale)\
                                       .clip(ROI)
        
        fpar_terra_max = dataset_terra.select('Fpar')\
                                       .max()\
                                       .multiply(0.01)\
                                       .rename('FPAR_Terra_max')\
                                       .reproject(crs=crs, scale=scale)\
                                       .clip(ROI)

        dataset_aqua = MCD15A3H.map(apply_bitmask_aqua)

        fpar_aqua_mean = dataset_aqua.select('Fpar')\
                                       .mean()\
                                       .multiply(0.01)\
                                       .rename('FPAR_Aqua_mean')\
                                       .reproject(crs=crs, scale=scale)\
                                       .clip(ROI)
        
        fpar_aqua_min = dataset_aqua.select('Fpar')\
                                       .min()\
                                       .multiply(0.01)\
                                       .rename('FPAR_Aqua_min')\
                                       .reproject(crs=crs, scale=scale)\
                                       .clip(ROI)
        
        fpar_aqua_max = dataset_aqua.select('Fpar')\
                                       .max()\
                                       .multiply(0.01)\
                                       .rename('FPAR_Aqua_max')\
                                       .reproject(crs=crs, scale=scale)\
                                       .clip(ROI)
        
        combined = esi_4wk_mean.addBands(esi_4wk_min)\
                               .addBands(esi_4wk_max)\
                               .addBands(fpar_terra_mean)\
                               .addBands(fpar_terra_min)\
                               .addBands(fpar_terra_max)\
                               .addBands(fpar_aqua_mean)\
                               .addBands(fpar_aqua_min)\
                               .addBands(fpar_aqua_max)\
                            #    .addBands(sm_root_mean)\
                            #    .addBands(sm_root_min)\
                            #    .addBands(sm_root_max)\
                            #    .addBands(et_mean)\
                            #    .addBands(et_min)\
                            #    .addBands(et_max)\

        
        feat_coll = combined.sample(factor=1,
                                    region=ROI,
                                    geometries=True,
                                    scale=scale,
                                    projection=crs)
                        
        def add_properties(feature):
            
            return feature.set({
                'Timestamp': START_DATE,
                'lon': feature.geometry().coordinates().get(0),
                'lat': feature.geometry().coordinates().get(1)
            })
        
        processed_fc = feat_coll.map(add_properties) 

        def add_properties(feature):
            
            return feature.set({
                'Timestamp': START_DATE,
                'lon': feature.geometry().coordinates().get(0),
                'lat': feature.geometry().coordinates().get(1)
            })
        
        processed_fc = feat_coll.map(add_properties)

        bands = ['Timestamp', 'lon', 'lat', 'ESI_4wk_mean', 'ESI_4wk_min', 'ESI_4wk_max',
                'ET_mean', 'ET_min', 'ET_max', 'SM_ROOT_mean', 'SM_ROOT_min', 'SM_ROOT_max',
                'FPAR_Terra_mean', 'FPAR_Terra_min', 'FPAR_Terra_max',
                'FPAR_Aqua_mean', 'FPAR_Aqua_min', 'FPAR_Aqua_max']
        
        # bands = ['Timestamp', 'lon', 'lat', 'ESI_4wk_mean', 'ESI_4wk_mean', 'ESI_4wk_mean']

        # task = ee.batch.Export.table.toDrive(collection=processed_fc,
        #                                     description=f'{tile}_{scale}m_{START_DATE}_maize',
        #                                     # description='test',
        #                                     fileFormat='CSV',
        #                                     selectors=bands,
        #                                     folder='maize_extra_metrics')

        # task.start()                   

CRS: EPSG:4326
scale: 100
Start: 2017-01. End: 2022-12
  N09E106_0_0
    2017-06


In [ ]:
START_DATE = months[0] + '-01'
STOP_DATE = get_last_day_of_month_from_Ym(months[0],
                                                input_format='%Y-%m',
                                                output_format='%Y-%m-%d')

dataset = ee.ImageCollection("MODIS/061/MCD15A3H").filterDate(START_DATE, STOP_DATE)

def extract_bits(image, bit_start, bit_end):
    num_bits = bit_end - bit_start + 1
    mask = (1 << num_bits) - 1
    return image.rightShift(bit_start).bitwiseAnd(mask)

def apply_bitmask_terra(image):

    qc_band = image.select('FparLai_QC')

    # Extract values for the specific bit ranges
    modland_qc = extract_bits(qc_band, 0, 0).eq(0)  # Bit 0 should be 0 (Good quality - main algorithm with or without saturation)
    detector = extract_bits(qc_band, 2, 2).eq(0)    # Bit 2 should be 0
    cloud_state = extract_bits(qc_band, 3, 4).eq(0) # Bits 3-4 should be 0 (clear)
    scf_qc = extract_bits(qc_band, 5, 7).neq(4)     # Bits 5-7 should not be 4 (Pixel not produced at all, value couldn''t be retrieved)

    combined_mask = modland_qc.And(detector).And(cloud_state).And(scf_qc)

    # Sensor
    terra_mask = combined_mask.And(extract_bits(qc_band, 1, 1).eq(0)) 

    return terra_mask

def apply_bitmask_aqua(image):

    qc_band = image.select('FparLai_QC')

    # Extract values for the specific bit ranges
    modland_qc = extract_bits(qc_band, 0, 0).eq(0)  # Bit 0 should be 0 (Good quality - main algorithm with or without saturation)
    detector = extract_bits(qc_band, 2, 2).eq(0)    # Bit 2 should be 0
    cloud_state = extract_bits(qc_band, 3, 4).eq(0) # Bits 3-4 should be 0 (clear)
    scf_qc = extract_bits(qc_band, 5, 7).neq(4)     # Bits 5-7 should not be 4 (Pixel not produced at all, value couldn''t be retrieved)

    combined_mask = modland_qc.And(detector).And(cloud_state).And(scf_qc)
    aqua_mask =  combined_mask.And(extract_bits(qc_band, 1, 1).eq(1)) 

    return aqua_mask

dataset_terra = dataset.map(apply_bitmask_terra)
dataset_aqua = dataset.map(apply_bitmask_aqua)

{'type': 'Image', 'bands': [{'id': 'FparLai_QC', 'data_type': {'type': 'PixelType', 'precision': 'int', 'min': 0, 'max': 255}, 'dimensions': [86400, 43200], 'crs': 'SR-ORG:6974', 'crs_transform': [463.3127165279165, 0, -20015109.353988, 0, -463.3127165274999, 10007554.676994]}], 'version': 1638759926615693, 'id': 'MODIS/006/MOD15A2H/2019_07_12', 'properties': {'system:time_start': 1562889600000, 'google:max_source_file_timestamp': 1563726613000, 'system:footprint': {'type': 'LinearRing', 'coordinates': [[-180, -90], [180, -90], [180, 90], [-180, 90], [-180, -90]]}, 'system:time_end': 1563580800000, 'system:asset_size': 1624113975, 'system:index': '2019_07_12'}}


In [1]:
from tqdm import tqdm_notebook
import glob
import pandas as pd

dfs = []
wheat_xtra_paths = glob.glob("G:\My Drive\Work\ADB\SAR\wheat_extra_metrics\*")
for path in tqdm_notebook(wheat_xtra_paths):
    df = pd.read_csv(path)
    dfs.append(df)

wheat_extra_df = pd.concat(dfs, axis=0).reset_index(drop=True)
wheat_extra_df

<>:6: SyntaxWarning: invalid escape sequence '\M'
<>:6: SyntaxWarning: invalid escape sequence '\M'
C:\Users\sonle\AppData\Local\Temp\ipykernel_20668\656395680.py:6: SyntaxWarning: invalid escape sequence '\M'
  wheat_xtra_paths = glob.glob("G:\My Drive\Work\ADB\SAR\wheat_extra_metrics\*")
C:\Users\sonle\AppData\Local\Temp\ipykernel_20668\656395680.py:7: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for path in tqdm_notebook(wheat_xtra_paths):


  0%|          | 0/792 [00:00<?, ?it/s]

,Timestamp,lon,lat,ESI_4wk_mean,ESI_4wk_min,ESI_4wk_max,ET_mean,ET_min,ET_max,SM_ROOT_mean,SM_ROOT_min,SM_ROOT_max,FPAR_Terra_mean,FPAR_Terra_min,FPAR_Terra_max,FPAR_Aqua_mean,FPAR_Aqua_min,FPAR_Aqua_max
0,2017-01-01,105.220118,10.002292,0.640430,0.357458,1.16956,34.333333,33.0,36.0,0.455556,0.419219,0.501966,0.763333,0.50,0.93,0.596000,0.06,0.92
1,2017-01-01,105.222813,10.002292,0.640430,0.357458,1.16956,34.333333,33.0,36.0,0.455556,0.419219,0.501966,0.763333,0.50,0.93,0.596000,0.06,0.92
2,2017-01-01,105.240780,10.002292,0.640430,0.357458,1.16956,39.666667,33.0,50.0,0.455556,0.419219,0.501966,0.647500,0.27,0.93,0.810000,0.57,0.93
3,2017-01-01,105.240780,10.003190,0.640430,0.357458,1.16956,39.666667,33.0,50.0,0.455556,0.419219,0.501966,0.647500,0.27,0.93,0.810000,0.57,0.93
4,2017-01-01,105.244373,10.007681,0.640430,0.357458,1.16956,37.333333,29.0,50.0,0.455556,0.419219,0.501966,0.625000,0.48,0.77,0.581667,0.06,0.94
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1268349,2022-12-01,106.484048,10.988642,1.336569,-0.391555,3.50000,NaN,NaN,NaN,0.367241,0.337713,0.391329,0.323333,0.24,0.38,0.400000,0.40,0.40
1268350,2022-12-01,106.282825,10.990438,0.796773,-0.063041,1.61694,NaN,NaN,NaN,0.402675,0.365911,0.430455,0.310000,0.22,0.39,0.395000,0.36,0.43
1268351,2022-12-01,106.256774,10.992235,0.796773,-0.063041,1.61694,NaN,NaN,NaN,0.402675,0.365911,0.430455,0.216667,0.12,0.32,0.220000,0.22,0.22
1268352,2022-12-01,106.308877,10.992235,0.744026,-0.133108,1.59349,NaN,NaN,NaN,0.402675,0.365911,0.430455,0.436667,0.38,0.50,0.490000,0.46,0.52


In [2]:
wheat_df = pd.read_csv(r"D:\Work\ADB\SAR\datasets\wheat_south.csv")
wheat_df

,CropType,Luminosity,NDVI_max,NDVI_mean,NDVI_min,PM25_mean,Precipitation_sum,Rel_humidity_mean,Temperature_max,Temperature_mean,Timestamp,elevation,lat,lon
0,Wheat,0.0,0.517756,0.353236,0.188716,11.338904,0.110568,0.813273,26.826220,25.706761,2017-01-01,2,9.500133,105.161728
1,Wheat,0.0,0.422222,0.272827,0.123431,11.338904,0.110568,0.813273,26.826220,25.706761,2017-01-01,1,9.501032,105.193169
2,Wheat,1.1,0.202432,0.151102,0.099772,11.338904,0.110568,0.813273,26.826220,25.706761,2017-01-01,0,9.501032,105.236288
3,Wheat,0.5,0.310885,0.223066,0.135248,11.338904,0.106291,0.809047,26.873258,25.760616,2017-01-01,2,9.501032,105.266831
4,Wheat,0.0,0.203149,0.174641,0.146133,11.338904,0.110568,0.813273,26.826220,25.706761,2017-01-01,1,9.501930,105.211135
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1551091,Wheat,0.0,0.685001,0.431208,0.103071,31.135925,0.056874,0.768715,27.135126,25.467303,2022-12-01,4,10.499958,106.006144
1551092,Wheat,0.8,0.742539,0.510616,0.100370,31.135925,0.056874,0.768715,27.135126,25.467303,2022-12-01,4,10.499958,106.016026
1551093,Wheat,0.0,0.751838,0.452165,0.090715,31.135925,0.056874,0.768715,27.135126,25.467303,2022-12-01,3,10.499958,106.030399
1551094,Wheat,1.7,0.692137,0.460408,0.182560,31.135925,0.056805,0.775177,27.134719,25.473841,2022-12-01,2,10.499958,106.103162


In [3]:
merged = pd.merge(wheat_df, wheat_extra_df, on=['Timestamp', 'lon', 'lat'], how='outer')
merged

,CropType,Luminosity,NDVI_max,NDVI_mean,NDVI_min,PM25_mean,Precipitation_sum,Rel_humidity_mean,Temperature_max,Temperature_mean,...,ET_max,SM_ROOT_mean,SM_ROOT_min,SM_ROOT_max,FPAR_Terra_mean,FPAR_Terra_min,FPAR_Terra_max,FPAR_Aqua_mean,FPAR_Aqua_min,FPAR_Aqua_max
0,Wheat,0.0,-0.014910,-0.067104,-0.119298,9.925525,0.081320,0.784391,27.662314,26.393133,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Wheat,0.0,0.635927,0.559682,0.483437,9.925525,0.070737,0.770593,27.657187,26.374345,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Wheat,0.0,0.098655,0.074111,0.049568,9.925525,0.061067,0.759146,27.759244,26.303365,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Wheat,0.8,0.127039,0.075331,0.023622,9.925525,0.081320,0.784391,27.662314,26.393133,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Wheat,0.0,0.156627,0.126228,0.095830,9.925525,0.081320,0.784391,27.662314,26.393133,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1723856,Wheat,7.9,0.721248,0.524188,0.032374,33.537773,0.043369,0.783757,27.366164,25.596966,...,NaN,0.484536,0.440252,0.522356,0.5150,0.44,0.58,0.513333,0.44,0.64
1723857,Wheat,2.5,0.425784,0.336742,0.086148,38.630043,0.040402,0.765052,27.306919,25.640070,...,NaN,0.500232,0.461833,0.530413,0.4825,0.38,0.56,0.540000,0.46,0.67
1723858,Wheat,13.2,0.259975,0.205334,0.103242,38.630043,0.038489,0.749907,27.485153,25.664438,...,NaN,0.498959,0.458751,0.529680,0.3520,0.28,0.40,0.385000,0.35,0.42
1723859,Wheat,3.2,0.536028,0.335517,0.126607,38.630043,0.038515,0.742740,27.521937,25.598336,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:

merged_nn = merged[(merged['ESI_4wk_mean'].notnull()) & (merged['ET_max'].notnull()) & (merged['NDVI_mean'].notnull())]
# merged_nn.to_csv(r'D:\Work\ADB\SAR\datasets/wheat_south_full_nonnull.csv', index=False)

In [23]:
merged.to_csv(r'D:\Work\ADB\SAR\datasets/wheat_south_full.csv', index=False)

In [5]:
dfs = []
wheat_sar_paths = glob.glob("D:\Work\ADB\SAR\wheat_sar/*")
for path in tqdm_notebook(wheat_sar_paths):
    df = pd.read_csv(path)
    dfs.append(df)

wheat_sar_df = pd.concat(dfs, axis=0).reset_index(drop=True)
wheat_sar_df

<>:2: SyntaxWarning: invalid escape sequence '\W'
<>:2: SyntaxWarning: invalid escape sequence '\W'
C:\Users\sonle\AppData\Local\Temp\ipykernel_20668\1543148896.py:2: SyntaxWarning: invalid escape sequence '\W'
  wheat_sar_paths = glob.glob("D:\Work\ADB\SAR\wheat_sar/*")
C:\Users\sonle\AppData\Local\Temp\ipykernel_20668\1543148896.py:3: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for path in tqdm_notebook(wheat_sar_paths):


  0%|          | 0/586 [00:00<?, ?it/s]

,RVI_max,RVI_mean,RVI_min,Timestamp,VHVV_max,VHVV_mean,VHVV_min,VH_max,VH_mean,VH_min,VVVH_max,VVVH_mean,VVVH_min,VV_max,VV_mean,VV_min,lat,lon
0,0.728416,0.629883,0.531350,2017-01-01,0.222649,0.187918,0.153186,0.042216,0.040131,0.038045,6.527999,5.509685,4.491371,0.248356,0.218983,0.189609,9.500133,105.161728
1,0.841539,0.768043,0.694547,2017-01-01,0.266440,0.238281,0.210122,0.026699,0.019266,0.011832,4.759151,4.256174,3.753197,0.127065,0.085737,0.044409,9.501032,105.193169
2,0.542118,0.524220,0.506322,2017-01-01,0.156778,0.150851,0.144925,0.024690,0.020920,0.017150,6.900110,6.639286,6.378463,0.170364,0.139877,0.109390,9.501032,105.236288
3,0.668881,0.644249,0.619616,2017-01-01,0.200798,0.192048,0.183298,0.036871,0.033021,0.029172,5.455606,5.217873,4.980141,0.201153,0.173217,0.145281,9.501032,105.266831
4,0.864847,0.827447,0.790046,2017-01-01,0.275855,0.260989,0.246124,0.045938,0.043307,0.040676,4.062994,3.844045,3.625095,0.166529,0.165897,0.165265,9.501930,105.211135
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1307152,0.466732,0.298932,0.131131,2022-12-01,0.132097,0.082995,0.033894,0.028045,0.018288,0.008532,29.503867,18.537045,7.570223,0.251725,0.232015,0.212304,10.499958,106.006144
1307153,0.751820,0.548449,0.345077,2022-12-01,0.231459,0.162937,0.094414,0.025176,0.024911,0.024647,10.591624,7.456021,4.320419,0.266650,0.186568,0.106485,10.499958,106.016026
1307154,0.771088,0.454487,0.137885,2022-12-01,0.238807,0.137255,0.035702,0.032656,0.020172,0.007688,28.009616,16.098547,4.187477,0.215325,0.176036,0.136748,10.499958,106.030399
1307155,0.660891,0.586382,0.511873,2022-12-01,0.197924,0.172336,0.146747,0.030770,0.029096,0.027422,6.814444,5.933438,5.052432,0.209678,0.174113,0.138547,10.499958,106.135502


In [6]:
merged_sar = pd.merge(merged, wheat_sar_df, on=['Timestamp', 'lat', 'lon'], how='left')
merged_sar.to_csv(r'D:\Work\ADB\SAR\datasets/wheat_south_sar_full.csv', index=False)


In [12]:
merged_sar_nn = pd.merge(merged_nn, wheat_sar_df, on=['Timestamp', 'lat', 'lon'], how='inner')
merged_sar.to_csv(r'D:\Work\ADB\SAR\datasets/wheat_south_sar_nnull.csv', index=False)

In [8]:
merged_sar

,CropType,Luminosity,NDVI_max,NDVI_mean,NDVI_min,PM25_mean,Precipitation_sum,Rel_humidity_mean,Temperature_max,Temperature_mean,...,VHVV_min,VH_max,VH_mean,VH_min,VVVH_max,VVVH_mean,VVVH_min,VV_max,VV_mean,VV_min
0,Wheat,0.0,-0.014910,-0.067104,-0.119298,9.925525,0.081320,0.784391,27.662314,26.393133,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Wheat,0.0,0.635927,0.559682,0.483437,9.925525,0.070737,0.770593,27.657187,26.374345,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Wheat,0.0,0.098655,0.074111,0.049568,9.925525,0.061067,0.759146,27.759244,26.303365,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Wheat,0.8,0.127039,0.075331,0.023622,9.925525,0.081320,0.784391,27.662314,26.393133,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Wheat,0.0,0.156627,0.126228,0.095830,9.925525,0.081320,0.784391,27.662314,26.393133,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1723856,Wheat,7.9,0.721248,0.524188,0.032374,33.537773,0.043369,0.783757,27.366164,25.596966,...,0.211211,0.057742,0.054913,0.052084,4.734609,4.272754,3.810899,0.273387,0.235937,0.198487
1723857,Wheat,2.5,0.425784,0.336742,0.086148,38.630043,0.040402,0.765052,27.306919,25.640070,...,0.169427,0.040230,0.036503,0.032777,5.902234,5.821321,5.740408,0.237445,0.212798,0.188151
1723858,Wheat,13.2,0.259975,0.205334,0.103242,38.630043,0.038489,0.749907,27.485153,25.664438,...,0.158243,0.036590,0.030523,0.024456,6.319385,5.423145,4.526906,0.165641,0.160093,0.154545
1723859,Wheat,3.2,0.536028,0.335517,0.126607,38.630043,0.038515,0.742740,27.521937,25.598336,...,0.135834,0.038016,0.037853,0.037690,7.361954,6.612080,5.862206,0.279873,0.250410,0.220948


In [11]:
merged_sar_nn

,CropType,Luminosity,NDVI_max,NDVI_mean,NDVI_min,PM25_mean,Precipitation_sum,Rel_humidity_mean,Temperature_max,Temperature_mean,...,VHVV_min,VH_max,VH_mean,VH_min,VVVH_max,VVVH_mean,VVVH_min,VV_max,VV_mean,VV_min
0,Wheat,0.0,0.546961,0.538698,0.526727,10.479693,0.110359,0.813718,26.864224,25.832589,...,0.144763,0.044193,0.043617,0.043041,6.907839,6.086418,5.264996,0.305280,0.265944,0.226608
1,Wheat,0.0,0.588834,0.506336,0.428089,10.479693,0.110359,0.813718,26.864224,25.832589,...,0.225356,0.045356,0.041786,0.038216,4.437433,4.108740,3.780047,0.171448,0.170514,0.169580
2,Wheat,0.0,0.616515,0.529869,0.443129,10.479693,0.110359,0.813718,26.864224,25.832589,...,0.251944,0.055666,0.050644,0.045621,3.969140,3.924087,3.879034,0.215932,0.198504,0.181075
3,Wheat,0.0,0.160550,0.131569,0.095159,10.479693,0.110359,0.813718,26.864224,25.832589,...,0.184077,0.016953,0.015926,0.014900,5.432504,5.111386,4.790268,0.081208,0.081076,0.080945
4,Wheat,0.0,0.259416,0.169800,0.034301,10.479693,0.110359,0.813718,26.864224,25.832589,...,0.242619,0.029783,0.028834,0.027885,4.121689,3.976978,3.832267,0.114935,0.114536,0.114137
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
820994,Wheat,3.8,0.231704,0.148541,0.010825,14.559668,0.307570,0.863732,28.809053,27.294521,...,0.083799,0.044412,0.044338,0.044264,11.933367,10.093640,8.253913,0.528219,0.447397,0.366576
820995,Wheat,1.5,0.350427,0.226757,0.094148,20.506948,0.302330,0.856314,28.799938,27.351840,...,0.171770,0.045828,0.041998,0.038168,5.821722,5.688909,5.556096,0.254623,0.238412,0.222201
820996,Wheat,5.0,0.510332,0.273863,0.038833,14.559668,0.307570,0.863732,28.809053,27.294521,...,0.222960,0.053326,0.051293,0.049261,4.485110,4.481135,4.477161,0.239173,0.229861,0.220548
820997,Wheat,1.4,0.370499,0.232256,0.107189,20.506948,0.302330,0.856314,28.799938,27.351840,...,0.169319,0.043405,0.042271,0.041136,5.906015,5.731253,5.556491,0.242950,0.242066,0.241182
